In [1]:
# Same parameter pattern as NB_01 and NB_02
# file_name is overridden by the pipeline ForEach at runtime

file_name     = "listings_apac.xlsx"
source_region = "APAC"
file_path     = f"Files/incoming/{file_name}"

print(f"Processing: {file_name}")
print(f"Source region: {source_region}")
print(f"Full path : {file_path}")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 3, Finished, Available, Finished, False)

Processing: listings_apac.xlsx
Source region: APAC
Full path : Files/incoming/listings_apac.xlsx


In [2]:
# WHY openpyxl AND NOT pandas.read_excel:
# pandas.read_excel internally uses openpyxl for .xlsx files anyway
# openpyxl gives us more control — we can read specific sheets,
# handle merged cells, and access cell formatting if needed
# In Fabric notebooks openpyxl is available without installation
# but we import it explicitly to make the dependency clear

import openpyxl
import pandas as pd
from notebookutils import mssparkutils
import io

print(f"openpyxl version: {openpyxl.__version__}")
print(f"pandas version: {pd.__version__}")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 4, Finished, Available, Finished, False)

openpyxl version: 3.0.10
pandas version: 2.1.4


In [5]:
# WHY WE READ VIA mssparkutils AND NOT DIRECT PATH:
# The file lives in the Lakehouse Files section which is
# an ADLS Gen2 path — not a local filesystem path.
# mssparkutils.fs.open() gives us a byte stream from that path
# which openpyxl can read directly — no local file copy needed.
#
# WHY CONVERT TO PANDAS FIRST:
# openpyxl reads row by row — slow for large files.
# Pandas gives us a clean DataFrame immediately.
# We then convert Pandas → Spark for the rest of the processing.
# This is the standard Excel ingestion pattern in Databricks and Fabric.

# Read file bytes from Lakehouse Files
#file_bytes = mssparkutils.fs.open(file_path, "rb").read()
# Use the local mount path for standard Python open()
# Fabric maps 'Files/' to '/lakehouse/default/Files/'

local_path = f"/lakehouse/default/{file_path}"
with open(local_path, "rb") as f:
    file_bytes = f.read()


# Load into openpyxl workbook
workbook    = openpyxl.load_workbook(io.BytesIO(file_bytes), data_only=True)
sheet_name  = workbook.sheetnames[0]
worksheet   = workbook[sheet_name]

print(f"Sheet name: {sheet_name}")
print(f"Total rows (including header): {worksheet.max_row}")
print(f"Total columns: {worksheet.max_column}")

# Read into Pandas DataFrame
rows = list(worksheet.values)
headers = rows[0]
data    = rows[1:]

df_pandas = pd.DataFrame(data, columns=headers)
print(f"\nPandas DataFrame: {len(df_pandas)} rows × {len(df_pandas.columns)} columns")
print(f"Columns: {list(df_pandas.columns)}")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 7, Finished, Available, Finished, False)

Sheet name: APAC Listings
Total rows (including header): 401
Total columns: 16

Pandas DataFrame: 400 rows × 16 columns
Columns: ['PropertyRef', 'ListingAgent', 'Type', 'CityName', 'Location', 'BedCount', 'BathCount', 'FloorArea_sqft', 'Price_USD', 'PricePerSqFt', 'DateListed', 'ListingStatus', 'ViewType', 'FurnishedStatus', 'Description', 'LastUpdated']


In [6]:
# WHY CONVERT TO SPARK:
# All downstream processing uses Spark — Delta writes, quality rules,
# joins. Staying in Pandas for a 400-row file is fine but we convert
# to Spark immediately to stay consistent with the rest of the pipeline.
#
# WHY schema inference is safe here:
# We just read from Excel where the data is already typed
# Pandas infers types correctly for small clean files
# We will cast explicitly in the next cell anyway

df_raw = spark.createDataFrame(df_pandas.astype(str))

# Convert all columns to string first — same defensive pattern as CSV
# openpyxl sometimes reads numbers as Python ints or floats
# astype(str) normalises everything before Spark casting

raw_count = df_raw.count()
print(f"Spark DataFrame: {raw_count} rows")
print("\nSchema:")
df_raw.printSchema()

df_raw.select(
    "PropertyRef", "CityName", "Location",
    "Price_USD", "DateListed", "ListingStatus"
).show(5, truncate=False)

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 8, Finished, Available, Finished, False)

Spark DataFrame: 400 rows

Schema:
root
 |-- PropertyRef: string (nullable = true)
 |-- ListingAgent: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- CityName: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- BedCount: string (nullable = true)
 |-- BathCount: string (nullable = true)
 |-- FloorArea_sqft: string (nullable = true)
 |-- Price_USD: string (nullable = true)
 |-- PricePerSqFt: string (nullable = true)
 |-- DateListed: string (nullable = true)
 |-- ListingStatus: string (nullable = true)
 |-- ViewType: string (nullable = true)
 |-- FurnishedStatus: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- LastUpdated: string (nullable = true)

+-----------+---------+---------+---------+----------+-------------+
|PropertyRef|CityName |Location |Price_USD|DateListed|ListingStatus|
+-----------+---------+---------+---------+----------+-------------+
|APAC-00001 |Kowloon  |Hong Kong|2075000  |2022/12/12|Sold         |
|APAC-00

In [7]:
from pyspark.sql.functions import (
    col, to_date, trim, upper, lower, initcap,
    when, lit, current_timestamp,
    round as spark_round
)

# COLUMN MAPPING — APAC Excel → Unified schema:
# PropertyRef      → ListingID
# ListingAgent     → AgentName
# Type             → PropertyType
# CityName         → City
# Location         → Country    (full name — no conversion needed)
# BedCount         → Bedrooms
# BathCount        → Bathrooms
# FloorArea_sqft   → AreaSqFt   (already sqft — no conversion needed)
# Price_USD        → ListingPrice
# PricePerSqFt     → PricePerSqFt  (keep — APAC-specific extra column)
# DateListed       → ListDate   (parse YYYY/MM/DD below)
# ListingStatus    → Status     (Available → Active)
# ViewType         → ViewType   (keep — APAC-specific extra column)
# FurnishedStatus  → FurnishedStatus (keep — APAC-specific extra column)
# Description      → Description
# LastUpdated      → dropped
#
# WHY KEEP EXTRA COLUMNS:
# Bronze should preserve as much source data as possible
# ViewType and FurnishedStatus are genuinely useful for APAC analytics
# They will be null for USA and Europe rows in the unified table
# Silver can choose whether to include them

df_renamed = df_raw \
    .withColumnRenamed("PropertyRef",    "ListingID") \
    .withColumnRenamed("ListingAgent",   "AgentName") \
    .withColumnRenamed("Type",           "PropertyType") \
    .withColumnRenamed("CityName",       "City") \
    .withColumnRenamed("Location",       "Country") \
    .withColumnRenamed("BedCount",       "Bedrooms") \
    .withColumnRenamed("BathCount",      "Bathrooms") \
    .withColumnRenamed("FloorArea_sqft", "AreaSqFt") \
    .withColumnRenamed("Price_USD",      "ListingPrice") \
    .withColumnRenamed("DateListed",     "ListDate") \
    .withColumnRenamed("ListingStatus",  "Status") \
    .drop("LastUpdated")

print("Columns after rename:")
print(df_renamed.columns)

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 9, Finished, Available, Finished, False)

Columns after rename:
['ListingID', 'AgentName', 'PropertyType', 'City', 'Country', 'Bedrooms', 'Bathrooms', 'AreaSqFt', 'ListingPrice', 'PricePerSqFt', 'ListDate', 'Status', 'ViewType', 'FurnishedStatus', 'Description']


In [8]:
# WHY THIS FORMAT IS DIFFERENT FROM BOTH OTHER FILES:
# USA:    MM/dd/yyyy  →  05/23/2022
# Europe: dd-MM-yyyy  →  23-05-2022
# APAC:   yyyy/MM/dd  →  2022/05/23
#
# All three represent the same date but Spark needs the exact format
# Using the wrong format produces null dates with no error
# Always verify by checking null count after parsing

df_dated = df_renamed \
    .withColumn("ListDate",
        to_date(col("ListDate"), "yyyy/MM/dd"))

null_dates = df_dated.filter(col("ListDate").isNull()).count()
print(f"Null dates after parsing: {null_dates}  (should be 0)")

df_dated.select("ListingID", "ListDate").show(5, truncate=False)

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 10, Finished, Available, Finished, False)

Null dates after parsing: 0  (should be 0)
+----------+----------+
|ListingID |ListDate  |
+----------+----------+
|APAC-00001|2022-12-12|
|APAC-00002|2024-03-24|
|APAC-00003|2022-03-21|
|APAC-00004|2023-03-11|
|APAC-00005|2022-10-29|
+----------+----------+
only showing top 5 rows



In [9]:
# AreaSqFt is already sqft in the APAC file
# No unit conversion needed unlike the Europe file

df_typed = df_dated \
    .withColumn("Bedrooms",
        col("Bedrooms").cast("integer")) \
    .withColumn("Bathrooms",
        col("Bathrooms").cast("integer")) \
    .withColumn("AreaSqFt",
        col("AreaSqFt").cast("double")) \
    .withColumn("ListingPrice",
        col("ListingPrice").cast("double")) \
    .withColumn("PricePerSqFt",
        col("PricePerSqFt").cast("double"))

# Verify no nulls from bad casts
print("Null check after casting:")
for c in ["Bedrooms", "Bathrooms", "AreaSqFt", "ListingPrice"]:
    null_count = df_typed.filter(col(c).isNull()).count()
    print(f"  {c}: {null_count} nulls")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 11, Finished, Available, Finished, False)

Null check after casting:
  Bedrooms: 0 nulls
  Bathrooms: 0 nulls
  AreaSqFt: 0 nulls
  ListingPrice: 0 nulls


In [10]:
# APAC STATUS MAPPING:
# Available → Active   (Available is APAC-specific term for Active)
# Sold      → Sold     (same meaning)
# any other → Other
#
# WHY Available maps to Active and not keeps as Available:
# The unified Silver schema uses Active/Sold/Other
# Available means the same thing as Active — the listing is live
# Mapping it ensures cross-regional analysis works correctly
# A query for "all active listings globally" returns APAC listings too

df_standardised = df_typed \
    .withColumn("Status",
        when(col("Status") == "Available", lit("Active"))
       .when(col("Status") == "Sold",      lit("Sold"))
       .otherwise(lit("Other")))

print("Status distribution after standardisation:")
df_standardised.groupBy("Status").count() \
    .orderBy("count", ascending=False).show()

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 12, Finished, Available, Finished, False)

Status distribution after standardisation:
+------+-----+
|Status|count|
+------+-----+
|Active|  259|
|  Sold|  141|
+------+-----+



In [11]:
# APAC uses: Apartment, House, Villa, Townhouse, Penthouse, Studio
# USA uses:  Single Family, Condo, Townhouse, Multi-Family, Penthouse, Villa
# Europe uses: flat, house, villa, townhouse, penthouse, bungalow
#
# WHY STANDARDISE PROPERTY TYPE IN BRONZE FOR APAC ONLY:
# We standardise what we can here — the Silver notebook
# will do a final unified standardisation across all regions
# For APAC: Studio → Apartment (no equivalent in other files)

df_prop = df_standardised \
    .withColumn("PropertyType",
        when(col("PropertyType") == "Studio", lit("Apartment"))
       .otherwise(initcap(trim(col("PropertyType")))))

print("Property type distribution:")
df_prop.groupBy("PropertyType").count() \
    .orderBy("count", ascending=False).show()

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 13, Finished, Available, Finished, False)

Property type distribution:
+------------+-----+
|PropertyType|count|
+------------+-----+
|   Apartment|  129|
|   Penthouse|   79|
|       House|   70|
|   Townhouse|   64|
|       Villa|   58|
+------------+-----+



In [12]:
# Add source audit columns
# State = null for APAC — same as Europe
# ListingURL = null for APAC — that column does not exist in the Excel file

df_bronze = df_prop \
    .withColumn("SourceRegion",   lit("APAC")) \
    .withColumn("SourceFile",     lit(file_name)) \
    .withColumn("_ingestion_ts",  current_timestamp()) \
    .withColumn("State",          lit(None).cast("string")) \
    .withColumn("ListingURL",     lit(None).cast("string"))

# Remove existing APAC rows — idempotent rerun safety
from delta.tables import DeltaTable

if spark.catalog.tableExists("bronze_listings"):
    target = DeltaTable.forName(spark, "bronze_listings")
    target.delete("SourceRegion = 'APAC'")
    print("Removed existing APAC rows")

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze_listings")

apac_count = spark.sql(
    "SELECT COUNT(*) FROM bronze_listings WHERE SourceRegion = 'APAC'"
).collect()[0][0]
total = spark.table("bronze_listings").count()
print(f"Appended {apac_count} APAC rows  |  Total: {total}")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 14, Finished, Available, Finished, False)

Removed existing APAC rows
Appended 400 APAC rows  |  Total: 1800


In [13]:
print("=== NB_03 — APAC Excel Verification ===")

# All 3 regions should now be present
spark.sql("""
    SELECT SourceRegion, COUNT(*) AS Rows
    FROM   bronze_listings
    GROUP BY SourceRegion
    ORDER BY Rows DESC
""").show()

# APAC-specific columns should only have values for APAC rows
spark.sql("""
    SELECT
        SourceRegion,
        COUNT(ViewType)       AS HasViewType,
        COUNT(FurnishedStatus) AS HasFurnished
    FROM bronze_listings
    GROUP BY SourceRegion
""").show()

# Confirm country names are full names
spark.sql("""
    SELECT Country, COUNT(*) AS Count
    FROM   bronze_listings
    WHERE  SourceRegion = 'APAC'
    GROUP BY Country
    ORDER BY Count DESC
""").show()

# Price range — APAC prices in USD
spark.sql("""
    SELECT
        ROUND(MIN(ListingPrice), 0) AS MinPrice,
        ROUND(MAX(ListingPrice), 0) AS MaxPrice,
        ROUND(AVG(ListingPrice), 0) AS AvgPrice
    FROM bronze_listings
    WHERE SourceRegion = 'APAC'
""").show()

# Sample rows including APAC-specific columns
spark.table("bronze_listings") \
    .filter("SourceRegion = 'APAC'") \
    .select("ListingID", "City", "Country",
            "Bedrooms", "AreaSqFt", "ListingPrice",
            "ListDate", "Status", "ViewType", "FurnishedStatus") \
    .show(5, truncate=False)

print(f"\nTotal bronze_listings: {spark.table('bronze_listings').count()}")
print("Expected: 1,800 (800 USA + 600 Europe + 400 APAC)")

StatementMeta(, 7f44a703-898f-4460-a717-89f97f3cf41d, 15, Finished, Available, Finished, False)

=== NB_03 — APAC Excel Verification ===
+------------+----+
|SourceRegion|Rows|
+------------+----+
|         USA| 800|
|      Europe| 600|
|        APAC| 400|
+------------+----+

+------------+-----------+------------+
|SourceRegion|HasViewType|HasFurnished|
+------------+-----------+------------+
|      Europe|          0|           0|
|         USA|          0|           0|
|        APAC|        400|         400|
+------------+-----------+------------+

+---------+-----+
|  Country|Count|
+---------+-----+
|Australia|  247|
|Hong Kong|   98|
|Singapore|   55|
+---------+-----+

+--------+---------+---------+
|MinPrice| MaxPrice| AvgPrice|
+--------+---------+---------+
|334000.0|5988000.0|3165590.0|
+--------+---------+---------+

+----------+---------+---------+--------+--------+------------+----------+------+-------------+-------------------+
|ListingID |City     |Country  |Bedrooms|AreaSqFt|ListingPrice|ListDate  |Status|ViewType     |FurnishedStatus    |
+----------+---------+-